In [0]:
%pip install great-expectations

In [0]:
import great_expectations as gx
from datetime import datetime

In [0]:
df_spark = spark.table('rfb.raw.raw_cnae')
count = df_spark.count()

In [0]:
context = gx.get_context(mode="ephemeral")
df_pandas = df_spark.limit(1_000_000).toPandas()

data_source = context.data_sources.add_pandas('cnae_ds')
data_asset = data_source.add_dataframe_asset(name='cnae_asset')
batch_definition = data_asset.add_batch_definition_whole_dataframe('cnae_batch_def')

suite = context.suites.add(gx.ExpectationSuite(name='rfb_raw_cnae_suite'))

suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column='codigo_cnae'))
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column='descricao_cnae'))

validation_definition = context.validation_definitions.add(
    gx.ValidationDefinition(
        name='raw_cnae_validation',
        data=batch_definition,
        suite=suite
    )
)

gx_results = validation_definition.run(batch_parameters={'dataframe': df_pandas})

In [0]:
test_results = {
    'date_time': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    'gx_result' : gx_results,
    'data_count' : count,
    'is_data_count_valid': is_data_count_valid
}
if gx_results.success:
    print("="*60)
    print("Raw Cnaes is valid.\n")
    print(f'Test Results: {test_results}')
    print("="*60)
else:
    print(f'Test Results: {test_results}')
    raise Exception(f'ERROR: Raw Cnaes is invalid')
